## Lab 01 / Step 1 : Self-Supervised Learning (SSL) of Large Language Models (LLMs)

## Task : From a context of previous words, predict the next word in the sequence

### Xavier Bresson

### Number of data points for GPT-3, 175B parameters
+ Step #1 : 300B tokens
+ Step #2 : 10k-100k pairs (prompt, response)
+ Step #3 : 100k-1M triples (prompt, positive response, negative response)
+ Step #4 : 10k-100k prompts

### Number of data points for this tutorial
+ Step #1 : 200M tokens
+ Step #2 : 200K pairs (prompt, response)
+ Step #3 : 100K triples (prompt, positive response, negative response)
+ Step #4 : 100 prompts

### Objectives
+ Step-by-step approach to self-supervised learning for LLMs
+ Implementation of word prediction with { previous word, bag-of-words, attention mechanism }
+ Implement Transformer architecture with single head, multiple heads, PE, RC, LN, MLP
+ Train with batch of sequences for fast training with GPU
+ Save the pre-trained LLM network for Step 2


In [1]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/IPAM_Mar26_codes/labs_lecture09'
    print(path_to_file)
    # move to Google Drive directory
    os.chdir(path_to_file)
    !pwd
    

In [2]:
# Libraries
import torch
print(torch.__version__)
import torch.nn as nn
import torch.optim as optim
import time
import matplotlib.pyplot as plt
import logging
logging.getLogger().setLevel(logging.CRITICAL) # remove warnings
import os, datetime


2.1.2+cu118


## Time stamp for save/load data


In [3]:
# save time stamp
time_stamp = datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S")

# check dataset folder exists
data_dir = os.path.join('dataset')
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# select a time stamp
use_saved_time_stamp = False
use_saved_time_stamp = True
if use_saved_time_stamp:
    time_stamp = '26-02-07--18-48-32' # pre-trained on GPU 
print('time_stamp:', time_stamp, '\n')


time_stamp: 26-02-07--18-48-32 



## Generate training sequences

The training sequences are arithmetic series parametrized by ($s$, $d$, $n$):
$$\textrm{for k} = 0, 1, 2, ..., n-1:$$
$$a_k = s + k * d$$
where $s$ is the staring value, $d$ is the common difference, and $n$ is the number of terms in the sequence.

For example, with ($s$, $d$, $n$)=(2, 5, 4), the generated sequence is
$$a_0 = 2$$
$$a_1 = 7$$
$$a_2 = 12$$
$$a_3 = 17$$

Sequence Constraint: We truncate the sequence if any term exceeds the value of 100, e.g. ($s$, $d$, $n$)=(90, 5, 6) will generate $90, 95, 100$ (no more terms).




In [4]:
# generate arithmetic series
max_value = 100 # maximum value in the sequence
def arithmetic_series(max_value, s, d, n):
    seq = []
    for i in range(n):
        v = s + i * d
        if v <= max_value:
            seq.append(v)
        else:
            break
    return seq 

# generate training data, i.e. a long sequence of tokens defined as
#  seq = [ 2, 4, 6, <SEP>, 14, 17, 20, 22, <SEP>, 6, 8, ... ]
save_training_data = False
save_training_data = True
if save_training_data:

    # parameters for arithmetic series
    s = torch.randint(low=0, high=max_value, size=(1,)).item() # starting integer of the series
    d = torch.randint(low=1, high=10, size=(1,)).item() # value of common difference
    n = torch.randint(low=5, high=15, size=(1,)).item() # number of element in the series
    print('max_value: %d, start_value: %d, common_difference: %d, number_of_terms: %d' % (max_value,s,d,n))
    seq = arithmetic_series(max_value,s,d,n)
    print('example of an arithmetic series:',seq)

    # generate and save a sequence of arithmetic series separated with token <SEP>
    len_dataset = 100 # debug
    # len_dataset = 200000000 # length of the sequence of arithmetic series, 200M sequences
    print('len_dataset:', len_dataset)
    seq = []
    separator_token = '<SEP>' # separator token between series
    start = time.time()
    while len(seq)<=len_dataset:
        s = torch.randint(low=0, high=max_value, size=(1,)).item() # starting integer of the series
        d = torch.randint(low=1, high=10, size=(1,)).item() # value of common difference
        n = torch.randint(low=5, high=15, size=(1,)).item() # number of element in the series
        series = arithmetic_series(max_value,s,d,n) # generate arithmetic series
        series_token = [str(i) for i in series] # convert seq of integers into seq of tokens w/ string type
        series_token.append(separator_token) # append separator token
        seq.extend(series_token) # append one generated series to the sequence
    seq_tokens = seq[:len_dataset] # truncate the sequence to satisfy "len_dataset" number of tokens
    print('len(seq) data: %d, time(min): %.3f' % (len(seq_tokens), (time.time()-start)/60) )

    # save training data
    save_file = data_dir + '/step1_SSL_original_training_set_' + time_stamp + '.pt'
    print('save_file:', save_file, '\n')
    torch.save([seq_tokens],save_file) # save the sequence

    # print
    print('number of tokens in the sequence :',len(seq_tokens))
    print('print first 50 tokens :',seq_tokens[:50],'\n')
    print(f'Time(min): {((time.time()-start)/60):.3f}') # Time(min): 7.82 / 200M sequences

else:

    # load data
    load_file = data_dir + '/step1_SSL_original_training_set_' + time_stamp + '.pt'
    print('load_file:', load_file, '\n')
    seq_tokens = torch.load(load_file)[0]
    print('number of tokens in the sequence :',len(seq_tokens))
    print('print first 50 tokens :',seq_tokens[:50])


max_value: 100, start_value: 44, common_difference: 9, number_of_terms: 12
example of an arithmetic series: [44, 53, 62, 71, 80, 89, 98]
len_dataset: 100
len(seq) data: 100, time(min): 0.000
save_file: dataset/step1_SSL_original_training_set_26-02-07--18-48-32.pt 

number of tokens in the sequence : 100
print first 50 tokens : ['4', '12', '20', '28', '36', '44', '52', '60', '68', '76', '84', '92', '100', '<SEP>', '75', '78', '81', '84', '87', '90', '93', '96', '99', '<SEP>', '18', '22', '26', '30', '34', '38', '42', '46', '50', '54', '<SEP>', '63', '67', '71', '75', '79', '83', '87', '<SEP>', '6', '8', '10', '12', '14', '16', '18'] 

Time(min): 0.000


## Compute the dictionary of tokens and convert the sequence of tokens to integers

In [5]:
save_dictionary = False
# save_dictionary = True
if save_dictionary:

    # create the dictionary of tokens by extracting unique tokens (words)
    load_file = data_dir + '/step1_SSL_original_training_set_' + time_stamp + '.pt'
    print('load_file:', save_file, '\n')
    print('number of tokens in the sequence :',len(seq_tokens),'\n')
    dictionary = []
    num_tokens = 0
    for token in seq_tokens:
        if token not in dictionary:
            dictionary.append(token)
            num_tokens += 1
    print('dictionary:',dictionary,'\n')
    print('num_tokens (unique):',num_tokens,'\n')

    # add tokens to the dictionary for step #2
    tokens_for_step2 = ['generate', 'an', 'arithmetic', 'series', 'with', 'terms', 'starting', 'value', \
                        'and', 'common', 'difference', 'Let', 'be', 'the', 'number', 'of', 'then', 'write', \
                        'make', 'a', 'type', 'which', 'starts', 'at', 'elements', '<PAD>', '<EOS>']
    for token in torch.arange(max_value).tolist(): tokens_for_step2.append(str(token))
    # update dictionary
    for token in tokens_for_step2:
        if token not in dictionary:
            dictionary.append(token); num_tokens += 1
    print('updated dictionary:',dictionary,'\n')
    print('num_tokens (unique):',num_tokens,'\n')

    # token2index : dict w/ key=token(str) and value=index(int)
    # index2token : dict w/ key=index(int) and value=token(str)
    token2index = { token:index for index,token in enumerate(dictionary) }
    index2token = { index:token for index,token in enumerate(dictionary) }
    print('token2index:', token2index,'\n')
    print('index2token:', index2token,'\n')

    # func_tokens2indices : function that converts token (str) to indices (int) for token embedding
    # func_indices2tokens : function that converts indices (int) to token (str)
    # func_str2tokens : function that converts a string into tokens (str)
    # func_tokens2str : function that converts tokens (str) to a string
    func_tokens2indices = lambda list_tokens: [token2index[token] for token in list_tokens] # ['Let', '5', 'be', 'the'] => [113, 46, 114, 115]
    func_indices2tokens = lambda list_ints: [index2token[integer] for integer in list_ints] # [113, 46, 114, 115] => ['Let', '5', 'be', 'the']
    func_str2tokens = lambda input_str: [token_str for token_str in input_str.split()]      # 'Let 5 be the' => ['Let', '5', 'be', 'the']
    func_tokens2str = lambda list_str: ' '.join(list_str)                                   # ['Let', '5', 'be', 'the'] => 'Let 5 be the'

    # example
    seq_token = seq_tokens[:10] # first tokens
    print('seq_token:', seq_token,'\n')
    seq_ind = func_tokens2indices(seq_token) # token (str) to indices (int)
    print('seq_ind:', seq_ind,'\n')
    seq_token = func_indices2tokens(seq_ind) # indices (int) to token (str)
    print('back to seq_token:', seq_token,'\n')

    # example
    seq_token = seq_tokens[-10:] # last tokens
    print('seq_token:', seq_token,'\n')
    seq_ind = func_tokens2indices(seq_token) # token (str) to indices (int)
    print('seq_ind:', seq_ind,'\n')
    seq_token = func_indices2tokens(seq_ind) # indices (int) to token (str)
    print('back to seq_token:', seq_token,'\n')

    # convert long seq from tokens to torch integers for training
    seq = torch.tensor(func_tokens2indices(seq_tokens))
    print('number of tokens in the sequence :',seq.size(0),'\n')

    # save dictionary and training data
    save_file_dictionary = data_dir + '/step1_SSL_dictionary_' + time_stamp + '.pt'
    print('save_file_dictionary:', save_file_dictionary, '\n')
    torch.save([dictionary, num_tokens, token2index, index2token], save_file_dictionary) # save dictionary of tokens
    save_file_seq = data_dir + '/step1_SSL_indices_training_set_' + time_stamp + '.pt'
    print('save_file_seq:', save_file_seq, '\n')
    torch.save([seq], save_file_seq) # save the sequence of integers

else:

    # load dictionary 
    load_file_dictionary = data_dir + '/step1_SSL_dictionary_' + time_stamp + '.pt'

    # load dictionary from gpu pre-training   
    file_dictionary = 'dataset/step1_SSL_dictionary_26-02-07--18-48-32_200M.pt' # GPU pre-trained
    print('file_dictionary:', file_dictionary, '\n')
    import os
    os.makedirs('dataset', exist_ok=True)
    if not os.path.exists(file_dictionary):
        print(f'Downloading {file_dictionary} ...')
        dropbox_url = "https://www.dropbox.com/scl/fi/jvb5xkk0kfzwrm7vzha3o/step1_SSL_dictionary_26-02-07-18-48-32_200M.pt?rlkey=tprax7nqqpvogo04fhe7m2okd&dl=1"
        !wget -q "{dropbox_url}" -O "{file_dictionary}"
        
    dictionary, num_tokens, token2index, index2token = torch.load(file_dictionary) # load dictionary of tokens
    
    # load training data
    load_file_seq = data_dir + '/step1_SSL_indices_training_set_' + time_stamp + '.pt'
    print('load_file_seq:', load_file_seq, '\n')
    seq = torch.load(load_file_seq)[0] # load the sequence of integers

    # print
    print('dictionary:',dictionary,'\n')
    print('num_tokens (unique):',num_tokens,'\n')
    print('token2index:', token2index,'\n')
    print('index2token:', index2token,'\n')
    print('number of tokens in the sequence :',len(seq),'\n')
    func_tokens2indices = lambda list_tokens: [token2index[token] for token in list_tokens] # ['Let', '5', 'be', 'the'] => [113, 46, 114, 115]
    func_indices2tokens = lambda list_ints: [index2token[integer] for integer in list_ints] # [113, 46, 114, 115] => ['Let', '5', 'be', 'the']
    func_str2tokens = lambda input_str: [token_str for token_str in input_str.split()]      # 'Let 5 be the' => ['Let', '5', 'be', 'the']
    func_tokens2str = lambda list_str: ' '.join(list_str)                                   # ['Let', '5', 'be', 'the'] => 'Let 5 be the'

    # example
    seq_token = seq_tokens[:10] # first tokens
    print('seq_token:', seq_token,'\n')
    seq_ind = func_tokens2indices(seq_token) # token (str) to indices (int)
    print('seq_ind:', seq_ind,'\n')
    seq_token = func_indices2tokens(seq_ind) # indices (int) to token (str)
    print('back to seq_token:', seq_token,'\n')


load_file_dictionary: dataset/step1_SSL_dictionary_26-02-07-18-48-32_200M.pt 

load_file_seq: dataset/step1_SSL_indices_training_set_26-02-07--18-48-32.pt 

dictionary: ['51', '57', '63', '69', '75', '81', '87', '93', '99', '<SEP>', '91', '1', '2', '3', '4', '5', '6', '7', '8', '9', '77', '79', '83', '85', '89', '48', '55', '62', '76', '90', '97', '29', '38', '47', '56', '65', '74', '92', '94', '100', '50', '64', '71', '78', '39', '95', '46', '53', '60', '67', '88', '13', '22', '31', '40', '49', '58', '41', '42', '43', '44', '30', '54', '15', '36', '72', '86', '96', '18', '19', '20', '21', '23', '24', '25', '26', '27', '28', '73', '98', '82', '52', '34', '66', '70', '68', '84', '37', '61', '11', '14', '17', '32', '59', '10', '12', '16', '80', '35', '33', '45', '0', 'generate', 'an', 'arithmetic', 'series', 'with', 'terms', 'starting', 'value', 'and', 'common', 'difference', 'Let', 'be', 'the', 'number', 'of', 'then', 'write', 'make', 'a', 'type', 'which', 'starts', 'at', 'elements', '<

## Create batch of sub-sequences

In [6]:
# prepare batch of sub-sequences of the long sequence of tokens
#
#                              seq_len = 11 tokens
#                  ------------------------------
# seq            = [ 2, 0, 6, 3, 2, 9, 5, 7, 3, 4, 9 ]
#                       |<= sample start_idx = 1 (randomly selected in [0,1,...,batch_length-1])
#                       -------
#                     batch_length = 3 tokens
#
# all_sub_sequences = [ [0, 6, 3],   |
#                       [2, 9, 5],   | number of all sub-sequences = 11/3 ≈ 3
#                       [7, 3, 4] ]  |
#                        -------
#                      batch_length
#
# original list_batch_idx = [0, 1, 2] (2 = the number of all sub-sequences - 1 )
#
# sample batch_idx = [2, 1] (batch_size=2)
#
# batch_seq      = [ [7, 3, 4],   |
#                    [0, 6, 3] ]  | batch_size = 2 batches
#                     -------
#                   batch_length
#
# batch_target =   [ [3, 4, 9],   |
#  = batch_seq + 1   [6, 3, 2] ]  | batch_size
#                     -------
#                   batch_length
#

def get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx, device='cpu'):
    # input  : list_batch_idx = [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14], size=[15] 
    # output : list_batch_idx = [ 8,  2,  0,  6,  1, 10, 14,  9, 13,  7,  4, 11], size=[12], with e.g. batch_size=3
    num_sub_sequences = list_batch_idx.size(0)
    if num_sub_sequences == 0: # no more sequences
        return torch.tensor([]), torch.tensor([]), torch.tensor([], dtype=torch.long)
    bs = min(batch_size, num_sub_sequences) # if fewer samples remain than batch_size
    # shuffle indices and split 
    perm = torch.randperm(num_sub_sequences).to(device)
    selected_indices = perm[:bs]
    remaining_indices = perm[bs:]
    batch_idx = list_batch_idx[selected_indices] # actual batch indices
    new_list_batch_idx = list_batch_idx[remaining_indices] # remaining pool
    # Calculate the starting position for every sequence in the batch at once
    chunk_starts = start_idx + batch_idx * batch_length # size=[batch_size], e.g. chunk_starts=[20, 74, 32]
    offsets = torch.arange(batch_length).unsqueeze(0).to(device) # [0, 1, 2, ..., batch_length-1], size=[batch_length]
    # Broadcast to create a full 2D index matrix
    input_indices = chunk_starts.unsqueeze(1) + offsets # input_indices[i,j]=chunk_starts[i]+j, size=[batch_size, batch_length]
    # input_indices = [ [20, 21, 22, 23, 24, 25], [74, 75, 76, 77, 78, 79], [32, 33, 34, 35, 36, 37] ]
    target_indices = input_indices + 1 # targets are just input_indices shifted by +1
    # target_indices = [ [21, 22, 23, 24, 25, 26], [75, 76, 77, 78, 79, 80], [33, 34, 35, 36, 37, 38] ]
    batch_seq = seq[input_indices]
    # batch_seq = [ [17, 18,  5, 19, 20,  5], [15,  5, 51, 23, 52, 19], [ 5, 25, 17, 26, 27, 18] ]
    target_seq = seq[target_indices]  
    # target_seq = [ [18,  5, 19, 20,  5, 21], [ 5, 51, 23, 52, 19, 53], [25, 17, 26, 27, 18,  5] ]
    return batch_seq, target_seq, new_list_batch_idx


# parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 3; batch_length = 6 # debug
# batch_size = 100; batch_length = 100 # GPU
num_subseq = seq_len // batch_length - 1 # number of subsequences (-1 to guarantee the target is not empty)
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_subseq) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# print to understand
original_seq = seq[start_idx:]
original_batch_seq = torch.stack([seq[start_idx+i*batch_length : start_idx+(i+1)*batch_length] for i in list_batch_idx])
original_batch_seq = torch.stack([seq[start_idx+i*batch_length : start_idx+(i+1)*batch_length] for i in list_batch_idx])
print('original list_batch_idx: \n',list_batch_idx, list_batch_idx.size(),'\n')
print('original seq from start_idx: \n',original_seq, original_seq.size(),'\n')
print('all sub_sequences: \n',original_batch_seq, original_batch_seq.size(),'\n')
# print to understand

# print to understand
for idx_batch in range(num_batch): # e.g. num_batch for one full epoch
    print('-'*50)
    print('idx_batch: ', idx_batch, '\n')
    print('list_batch_idx (before): \n',list_batch_idx, list_batch_idx.size(),'\n')
    batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
    print('batch_seq:               \n',batch_seq, batch_seq.size(),'\n')
    print('target_seq:              \n',target_seq, target_seq.size(),'\n')
    print('list_batch_idx (after):  \n',list_batch_idx, list_batch_idx.size(),'\n')
    # break

    

seq_len: 100, batch_size: 3, batch_length: 6, num_subseq: 15, num_batch: 5

original list_batch_idx: 
 tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14]) torch.Size([15]) 

original seq from start_idx: 
 tensor([ 5,  4,  6,  7,  8,  9, 10,  5, 11, 12, 13, 14,  4, 15, 16, 17, 18,  5,
        19, 20,  5, 21, 22, 23, 17, 24, 20,  5, 25, 17, 26, 27, 18,  5, 28, 29,
        30, 31, 32,  6,  5, 33, 34, 35, 36, 37,  5, 38, 39, 40, 36, 41, 42, 28,
        11, 43, 44,  5, 45, 46, 14, 32, 47, 48, 49,  6,  8, 10, 50, 15,  5, 51,
        23, 52, 19, 53, 20,  5, 54, 55,  0, 56, 57, 58,  3, 59,  9, 60, 61, 62,
        63,  5, 64, 49,  7]) torch.Size([95]) 

all sub_sequences: 
 tensor([[ 5,  4,  6,  7,  8,  9],
        [10,  5, 11, 12, 13, 14],
        [ 4, 15, 16, 17, 18,  5],
        [19, 20,  5, 21, 22, 23],
        [17, 24, 20,  5, 25, 17],
        [26, 27, 18,  5, 28, 29],
        [30, 31, 32,  6,  5, 33],
        [34, 35, 36, 37,  5, 38],
        [39, 40, 36, 41, 42, 28],
   

## Stage 1 : Define class of token embedding

#### Token vector : $h_t \in \mathbb{R}^d$


In [7]:
# token embedding layer : convert seq of integers to seq of d-dim vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), 
                                                     # and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# print example
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
list_batch_idx = torch.arange(num_batch) # size=[batch_size]
batch_int, _, _ = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch, size=[batch_size, batch_length]
print('batch_int :',batch_int.size())

token2vec_layer = token2vec(num_tokens, d=128)
batch_vec = token2vec_layer(batch_int) # size=[batch_size, batch_length, d=128]
print('batch_vec :',batch_vec.size())


batch_int : torch.Size([3, 6])
batch_vec : torch.Size([3, 6, 128])


## Stage 2 : Vanilla Prediction 

#### Predict next token t+1 given the context = {current token : token t}, i.e. $h_{t+1} = f_\theta(h_{t})$


In [8]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

#    seq = [ 2, 3, 4, 5, 6 ]
#               - <= context = { current token } = 3
#               | <= predict next token = 4
# target = [ 3, 4, 5, 6, 7 ]
#               | <= score vector v must predict token "4"
# scores = [ v, v, v, v, v ]
#
# batch_seq      = [ [ 2, 3, 4, 5, 6 ],   |
#                    [ 2, 9, 5, 7, 3 ],   | batch_size
#                    [ 0, 6, 3, 5, 2 ] ]  |
#                     --------------
#                       batch_length
#
# batch_scores =   [ [ v, v, v, v, v ],   |
#                    [ v, v, v, v, v ],   | batch_size, v is a vector of "num_tokens" dimensions
#                    [ v, v, v, v, v ] ]  |
#                     --------------
#                      batch_length
#
# batch_target =   [ [ 3, 4, 5, 6, 7 ],   |
#  = batch_seq + 1   [ 9, 5, 7, 3, 1 ],   | batch_size
#                    [ 6, 3, 5, 2, 8 ] ]  |
#                     --------------
#                      batch_length

class vanillaLM(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        batch_seq_vec = self.token2vec(batch_seq) # size=[batch_size, batch_length, d]
        batch_scores = self.token_prediction(batch_seq_vec) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token

# batching parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 3; batch_length = 6 # debug
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# network parameters
d = 128 # embedding dimension
print('num_tokens: %d, d: %d\n' % (num_tokens, d) )
vanillaLLMnet = vanillaLM(num_tokens, d)
num_param = number_param(vanillaLLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Train network to predict next token
optimizer = torch.optim.AdamW(vanillaLLMnet.parameters(), lr=3e-4) # standard optimizer for LMs
num_epochs = 201 # 101(debug), number of epochs
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for _ in range(num_batch): # number of batches into one epoch
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
        batch_scores = vanillaLLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    loss_epoch = running_loss / num_batch
    if not epoch%10:
        print('Epoch: %d, time(sec): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, time.time()-start, optimizer.param_groups[0]['lr'], loss_epoch) )


seq_len: 100, batch_size: 3, batch_length: 6, num_subseq: 16, num_batch: 5

num_tokens: 129, d: 128

num_net_parameters: 33153 / 0.03 million

Epoch: 0, time(sec): 0.096, lr= 0.000300, loss_epoch: 4.951
Epoch: 10, time(sec): 0.155, lr= 0.000300, loss_epoch: 4.237
Epoch: 20, time(sec): 0.212, lr= 0.000300, loss_epoch: 3.597
Epoch: 30, time(sec): 0.270, lr= 0.000300, loss_epoch: 3.019
Epoch: 40, time(sec): 0.329, lr= 0.000300, loss_epoch: 2.513
Epoch: 50, time(sec): 0.387, lr= 0.000300, loss_epoch: 2.098
Epoch: 60, time(sec): 0.445, lr= 0.000300, loss_epoch: 1.785
Epoch: 70, time(sec): 0.503, lr= 0.000300, loss_epoch: 1.497
Epoch: 80, time(sec): 0.562, lr= 0.000300, loss_epoch: 1.286
Epoch: 90, time(sec): 0.621, lr= 0.000300, loss_epoch: 1.171
Epoch: 100, time(sec): 0.680, lr= 0.000300, loss_epoch: 0.999
Epoch: 110, time(sec): 0.738, lr= 0.000300, loss_epoch: 0.949
Epoch: 120, time(sec): 0.798, lr= 0.000300, loss_epoch: 0.883
Epoch: 130, time(sec): 0.857, lr= 0.000300, loss_epoch: 0.834


## Stage 3 : Bag-Of-Tokens Prediction

#### Predict next token t+1 given the context = { bag of tokens : tokens ≤ t }, i.e. $h_{t+1} = f_\theta(h_{\leq t})=f_\theta(\textrm{mean}_{\leq t}\{ h_{t} \})$

#### Aggregator of tokens is the mean operator (as a bag-of-token is a set of unordered tokens)


In [9]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

#    seq = [ 2, 3, 4, 5, 6 ]
#            ---------- <= context = { tokens(<=t) } = 2,3,4,5
#                     | <= predict next token = 6
# target = [ 3, 4, 5, 6, 7 ]
#                     | <= score vector v must predict token "6"
# scores = [ v, v, v, v, v ]
#
# triu(ones(3,3)) = [ [1, 0, 0]   = hide the future tokens
#                     [1, 1, 0]
#                     [1, 1, 1] ]
#
# mean_operator @ batch_seq_vec = [ [1, 0, 0]              [ [v1, v2]          [ [v1          , v2]
#                                   [1/2, 1/2, 0]       @    [v3, v4]     =      [(v1+v3)/2   , (v2+v4)/2]
#                                   [1/3, 1/3, 1/3] ]        [v5, v6] ]          [(v1+v3+v5)/3, (v2+v4+v6)/2] ]
#
# mean_operator @ batch_seq_vec = mean of tokens ≤ t vectors to predic the next token t+1 (bag-of-tokens)

class BOT_LM(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        batch_size = batch_seq.size(0); batch_len = batch_seq.size(1)
        batch_seq_vec = self.token2vec(batch_seq) # size=[batch_size, batch_length, d]
        mean_operator = torch.tril(torch.ones(batch_len,batch_len)).long() # mask to use previous tokens only : { token(<=t) }, size=[batch_len,batch_len]
        mean_operator = mean_operator/ torch.sum(mean_operator, dim=1).unsqueeze(1) # normalize w.r.t. number of previous tokens
        mean_operator = mean_operator.repeat(batch_size,1,1) # repeat masks batch_size times, size=(batch_size, batch_len, batch_len)
        batch_seq_vec =  mean_operator @ batch_seq_vec # matrix-matrix multiplication (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, num_tokens)
        batch_scores = self.token_prediction(batch_seq_vec) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token


# batching parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 3; batch_length = 6 # debug
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# network parameters
d = 128 # embedding dimension
print('num_tokens: %d, d: %d\n' % (num_tokens, d) )
BOT_LLMnet = BOT_LM(num_tokens, d)
num_param = number_param(BOT_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Train network to predict next token
optimizer = torch.optim.AdamW(BOT_LLMnet.parameters(), lr=3e-4) # standard optimizer for LMs
num_epochs = 201 # 101(debug), number of epochs
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for _ in range(num_batch): # number of batches into one epoch
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
        batch_scores = BOT_LLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    loss_epoch = running_loss / num_batch
    if not epoch%10:
        print('Epoch: %d, time(sec): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, time.time()-start, optimizer.param_groups[0]['lr'], loss_epoch) )


seq_len: 100, batch_size: 3, batch_length: 6, num_subseq: 16, num_batch: 5

num_tokens: 129, d: 128

num_net_parameters: 33153 / 0.03 million

Epoch: 0, time(sec): 0.008, lr= 0.000300, loss_epoch: 4.882
Epoch: 10, time(sec): 0.077, lr= 0.000300, loss_epoch: 4.486
Epoch: 20, time(sec): 0.146, lr= 0.000300, loss_epoch: 4.238
Epoch: 30, time(sec): 0.215, lr= 0.000300, loss_epoch: 3.941
Epoch: 40, time(sec): 0.284, lr= 0.000300, loss_epoch: 3.645
Epoch: 50, time(sec): 0.349, lr= 0.000300, loss_epoch: 3.400
Epoch: 60, time(sec): 0.412, lr= 0.000300, loss_epoch: 3.113
Epoch: 70, time(sec): 0.476, lr= 0.000300, loss_epoch: 2.877
Epoch: 80, time(sec): 0.539, lr= 0.000300, loss_epoch: 2.742
Epoch: 90, time(sec): 0.604, lr= 0.000300, loss_epoch: 2.512
Epoch: 100, time(sec): 0.668, lr= 0.000300, loss_epoch: 2.344
Epoch: 110, time(sec): 0.733, lr= 0.000300, loss_epoch: 2.274
Epoch: 120, time(sec): 0.798, lr= 0.000300, loss_epoch: 2.115
Epoch: 130, time(sec): 0.863, lr= 0.000300, loss_epoch: 1.978


## Stage 4 : Vanilla Self-Attention  

#### Predict next token t+1 given the context = { tokens ≤ t }

#### Aggregator of tokens is the self-attention operator : 

$H \ \leftarrow \ \textrm{Softmax}( HH^T / \sqrt{d} \ \odot \ \textrm{Mask} ) H$

$h_{t+1} \ \leftarrow \ \sum_{\leq t} \frac{\exp(h_t^T h_{t'})}{ \sum_{\leq t'} \exp(h_t^T h_{t'}) } h_t$


In [10]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

#    seq = [ 2, 3, 4, 5, 6 ]
#            ---------- <= context = { tokens(<=t) } = 2,3,4,5
#                     | <= predict next token = 6
# target = [ 3, 4, 5, 6, 7 ]
#                     | <= score vector v must predict token "6"
# scores = [ v, v, v, v, v ]
#
# triu(ones(3,3)) = [ [1, 0, 0]
#                     [1, 1, 0]
#                     [1, 1, 1] ]
#
# H = [ [ -- h1 -- ]    h = d-dim vectors
#       [ -- h2 --]
#       [ -- h3 -- ] ]
#
# attention_score = H^T H = [ [h1^2, h1h2, h1h3]   = similarities between token hi and token hj 
#                             [h2h1, h2^2, h2h3]
#                             [h3h1, h3h2, h3^2] ]
#
# attention_score.masked_fill = [ [h1^2,   -∞,   -∞]   = hide future tokens
#                                 [h2h1, h2^2,   -∞]
#                                 [h3h1, h3h2, h3^2] ]
#
# softmax(att_score.masked) = [ [s11,   0,   0]   = sij = probabilities of similarity between token hi and token hj 
#                               [s12, s22,   0]
#                               [s13, s23, s33] ]
#
# softmax @ H = [ [s11,   0,   0]         [ [h1]            [ [s11.h1] 
#                 [s12, s22,   0]     @     [h2]      =       [s12.h1 + s22.h1] 
#                 [s13, s23, s33] ]         [h3] ]            [s13.h1 + s23.h2 + s33.h3] ] 
#
#             = linear combination of token vectors, weigthed by the attention weight (similarity between tokens)

class VSA_LLM(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        batch_size = batch_seq.size(0); batch_len = batch_seq.size(1)
        H = self.token2vec(batch_seq) # size=[batch_size, batch_length, d]
        attention_score = H @ H.transpose(2,1) * H.size(2)**-0.5 # HH^T/sqrt(d), (B,L,d) @ (B,d,L) => (B,L,L), size=[batch_size, batch_length, batch_length)
        mask = torch.tril(torch.ones(batch_len,batch_len)).long() # mask to use previous tokens only : { token(<=t) }, size=[batch_len,batch_len]
        attention_score = attention_score.masked_fill(mask==0, value=float('-inf')) # softmax(-inf)=0 prevents using next tokens for prediction, size=(batch_size, batch_len, batch_len)
        attention_score = torch.softmax(attention_score, dim=2) # sum weights = 1, size=[batch_size, batch_length, batch_len)
        batch_seq_vec = attention_score @ H # softmax( HH^T / sqrt(d) ) H, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d)
        batch_scores = self.token_prediction(batch_seq_vec) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token

# batching parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 5; batch_length = 20 # bebug
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# network parameters
d = 128 # embedding dimension
print('num_tokens: %d, d: %d\n' % (num_tokens, d) )
VSA_LLMnet = VSA_LLM(num_tokens, d)
num_param = number_param(VSA_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Train network to predict next token
optimizer = torch.optim.AdamW(VSA_LLMnet.parameters(), lr=3e-4) # standard optimizer for LMs
num_epochs = 201 # 101(debug), number of epochs
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for _ in range(num_batch): # number of batches into one epoch
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
        batch_scores = VSA_LLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    loss_epoch = running_loss / num_batch
    if not epoch%10:
        print('Epoch: %d, time(sec): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, time.time()-start, optimizer.param_groups[0]['lr'], loss_epoch) )


seq_len: 100, batch_size: 5, batch_length: 20, num_subseq: 5, num_batch: 1

num_tokens: 129, d: 128

num_net_parameters: 33153 / 0.03 million

Epoch: 0, time(sec): 0.002, lr= 0.000300, loss_epoch: 4.962
Epoch: 10, time(sec): 0.020, lr= 0.000300, loss_epoch: 4.677
Epoch: 20, time(sec): 0.036, lr= 0.000300, loss_epoch: 4.393
Epoch: 30, time(sec): 0.053, lr= 0.000300, loss_epoch: 4.124
Epoch: 40, time(sec): 0.070, lr= 0.000300, loss_epoch: 3.851
Epoch: 50, time(sec): 0.087, lr= 0.000300, loss_epoch: 3.679
Epoch: 60, time(sec): 0.104, lr= 0.000300, loss_epoch: 3.342
Epoch: 70, time(sec): 0.121, lr= 0.000300, loss_epoch: 3.200
Epoch: 80, time(sec): 0.138, lr= 0.000300, loss_epoch: 2.983
Epoch: 90, time(sec): 0.155, lr= 0.000300, loss_epoch: 2.806
Epoch: 100, time(sec): 0.172, lr= 0.000300, loss_epoch: 2.477
Epoch: 110, time(sec): 0.188, lr= 0.000300, loss_epoch: 2.249
Epoch: 120, time(sec): 0.206, lr= 0.000300, loss_epoch: 2.161
Epoch: 130, time(sec): 0.222, lr= 0.000300, loss_epoch: 2.137


## Stage 5 : Standard Self-Attention 

#### Predict next token t+1 given the context = { tokens ≤ t }

#### Aggregator of tokens is the self-attention operator : 

$H \ \leftarrow \ \textrm{Softmax}( QK^T / \sqrt{d} ) V$,

with learnable dictionary $Q=W_Q H$ (Query), $K=W_K H$ (Key), and $V=W_V H$ (Value).

$h_{t+1} \ \leftarrow \ \sum_{\leq t} \frac{\exp(q_t^T k_{t'})}{ \sum_{\leq t'} \exp(q_t^T k_{t'}) } v_t$



In [11]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

class SA_LLM(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.query = nn.Linear(d, d, bias=False) # query embedding layer
        self.key = nn.Linear(d, d, bias=False) # key embedding layer
        self.value = nn.Linear(d, d) # value embedding layer
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        batch_size = batch_seq.size(0); batch_len = batch_seq.size(1)
        H = self.token2vec(batch_seq) # size=[batch_size, batch_length, d]
        Q = self.query(H) # size=[batch_size, batch_length, d]
        K = self.key(H) # size=[batch_size, batch_length, d]
        V = self.value(H) # size=[batch_size, batch_length, d]
        attention_score = Q @ K.transpose(2,1) * H.size(2)**-0.5 # QK^T/sqrt(d), (B,L,d) @ (B,d,L) => (B,L,L), size=[batch_size, batch_length, batch_length)
        mask = torch.tril(torch.ones(batch_len,batch_len)).long() # mask to use previous tokens only : { token(<=t) }, size=[batch_len,batch_len]
        attention_score = attention_score.masked_fill(mask==0, value=float('-inf')) # softmax(-inf)=0 prevents using next tokens for prediction, size=(batch_size, batch_len, batch_len)
        attention_score = torch.softmax(attention_score, dim=2) # sum weights = 1, size=[batch_size, batch_length, batch_len)
        batch_seq_vec = attention_score @ V # softmax( QK^T / sqrt(d) ) V, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d)
        batch_scores = self.token_prediction(batch_seq_vec) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token

# batching parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 5; batch_length = 20 # bebug
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# network parameters
d = 128 # embedding dimension
print('num_tokens: %d, d: %d\n' % (num_tokens, d) )
SA_LLMnet = SA_LLM(num_tokens, d)
num_param = number_param(SA_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Train network to predict next token
optimizer = torch.optim.AdamW(SA_LLMnet.parameters(), lr=3e-4) # standard optimizer for LMs
num_epochs = 201 # 101(debug), number of epochs
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for _ in range(num_batch): # number of batches into one epoch
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
        batch_scores = SA_LLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    loss_epoch = running_loss / num_batch
    if not epoch%10:
        print('Epoch: %d, time(sec): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, time.time()-start, optimizer.param_groups[0]['lr'], loss_epoch) )


seq_len: 100, batch_size: 5, batch_length: 20, num_subseq: 5, num_batch: 1

num_tokens: 129, d: 128

num_net_parameters: 82433 / 0.08 million

Epoch: 0, time(sec): 0.004, lr= 0.000300, loss_epoch: 4.851
Epoch: 10, time(sec): 0.030, lr= 0.000300, loss_epoch: 4.701
Epoch: 20, time(sec): 0.057, lr= 0.000300, loss_epoch: 4.483
Epoch: 30, time(sec): 0.083, lr= 0.000300, loss_epoch: 4.233
Epoch: 40, time(sec): 0.109, lr= 0.000300, loss_epoch: 3.895
Epoch: 50, time(sec): 0.134, lr= 0.000300, loss_epoch: 3.631
Epoch: 60, time(sec): 0.160, lr= 0.000300, loss_epoch: 3.345
Epoch: 70, time(sec): 0.186, lr= 0.000300, loss_epoch: 3.154
Epoch: 80, time(sec): 0.212, lr= 0.000300, loss_epoch: 2.797
Epoch: 90, time(sec): 0.238, lr= 0.000300, loss_epoch: 2.493
Epoch: 100, time(sec): 0.264, lr= 0.000300, loss_epoch: 2.206
Epoch: 110, time(sec): 0.292, lr= 0.000300, loss_epoch: 2.215
Epoch: 120, time(sec): 0.318, lr= 0.000300, loss_epoch: 1.775
Epoch: 130, time(sec): 0.344, lr= 0.000300, loss_epoch: 1.884


## Stage 6 : Add Positional Encoding (PE) to Self-Attention (SA)

#### Predict next token t+1 given the context = { tokens ≤ t }

#### Aggregator of tokens is the self-attention operator : 

$H \ \leftarrow \ \textrm{Token-Embedding} + \textrm{PE} \ \textrm{ (initial embedding)}$,

where PE is required to provide ordering information to the sequence of tokens.

$H \ \leftarrow \ \textrm{Softmax}( QK^T / \sqrt{d} ) V$,

with learnable $Q=W_Q H$ (Query), $K=W_K H$ (Key), $V=W_V H$ (Value).

$h_{t+1} \ \leftarrow \ \sum_{\leq t} \frac{\exp(q_t^T k_{t'})}{ \sum_{\leq t'} \exp(q_t^T k_{t'}) } v_t$




In [12]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# single head attention layer
class head_attention(nn.Module):
    def __init__(self, d, context_length):
        super().__init__()
        self.query = nn.Linear(d, d, bias=False) # query embedding layer
        self.key = nn.Linear(d, d, bias=False) # key embedding layer
        self.value = nn.Linear(d, d) # value embedding layer
        self.mask = torch.tril(torch.ones(context_length, context_length)).long() # mask to use previous tokens only : { token(<=t) }, size=[context_length, context_length]
    def forward(self, H):
        Q = self.query(H) # size=[batch_size, batch_length, d]
        K = self.key(H) # size=[batch_size, batch_length, d]
        V = self.value(H) # size=[batch_size, batch_length, d]
        attention_score = Q @ K.transpose(2,1) * H.size(2)**-0.5 # QK^T/sqrt(d), (B,L,d) @ (B,d,L) => (B,L,L), size=[batch_size, batch_length, batch_length)
        attention_score = attention_score.masked_fill(self.mask==0, value=float('-inf')) # softmax(-inf)=0 prevents using next tokens for prediction, size=(batch_size, batch_len, batch_len)
        attention_score = torch.softmax(attention_score, dim=2) # sum weights = 1, size=[batch_size, batch_length, batch_len]
        H_head = attention_score @ V # softmax( QK^T / sqrt(d) ) V, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d]
        return H_head

#             seq = [ 8, 4, 1, 5, 2, .. ]
#                     |  |  |  |  |
#    pos_encoding = [ 0, 1, 2, 3, 4, .. ] <-- Ordering of the tokens
#

class PE_LM(nn.Module):
    def __init__(self, num_tokens, d, context_length):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.HA = head_attention(d, context_length) # self-attention layer
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        seq_pos_encoding = torch.arange(batch_seq.size(1)) # positional encoding = {0,1,2,...,batch_length-1}
        H = self.token2vec(batch_seq) + self.PE_embedding(seq_pos_encoding).unsqueeze(0) # size=[batch_size, batch_length, d]
        batch_seq_vec = self.HA(H) # (single) attention head, size=[batch_size, batch_length, d]
        batch_scores = self.token_prediction(batch_seq_vec) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token

# batching parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 5; batch_length = 20 # bebug
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# network parameters
d = 128 # embedding dimension
print('num_tokens: %d, d: %d, batch_length: %d\n' % (num_tokens, d, batch_length) )
PE_LLMnet = PE_LM(num_tokens, d, batch_length)
num_param = number_param(PE_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Train network to predict next token
optimizer = torch.optim.AdamW(PE_LLMnet.parameters(), lr=3e-4) # standard optimizer for LMs
num_epochs = 201 # 101(debug), number of epochs
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for _ in range(num_batch): # number of batches into one epoch
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
        batch_scores = PE_LLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    loss_epoch = running_loss / num_batch
    if not epoch%10:
        print('Epoch: %d, time(sec): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, time.time()-start, optimizer.param_groups[0]['lr'], loss_epoch) )


seq_len: 100, batch_size: 5, batch_length: 20, num_subseq: 5, num_batch: 1

num_tokens: 129, d: 128, batch_length: 20

num_net_parameters: 84993 / 0.08 million

Epoch: 0, time(sec): 0.003, lr= 0.000300, loss_epoch: 4.868
Epoch: 10, time(sec): 0.030, lr= 0.000300, loss_epoch: 4.703
Epoch: 20, time(sec): 0.057, lr= 0.000300, loss_epoch: 4.469
Epoch: 30, time(sec): 0.084, lr= 0.000300, loss_epoch: 4.218
Epoch: 40, time(sec): 0.111, lr= 0.000300, loss_epoch: 3.950
Epoch: 50, time(sec): 0.138, lr= 0.000300, loss_epoch: 3.917
Epoch: 60, time(sec): 0.165, lr= 0.000300, loss_epoch: 3.727
Epoch: 70, time(sec): 0.192, lr= 0.000300, loss_epoch: 3.620
Epoch: 80, time(sec): 0.219, lr= 0.000300, loss_epoch: 3.147
Epoch: 90, time(sec): 0.246, lr= 0.000300, loss_epoch: 3.137
Epoch: 100, time(sec): 0.273, lr= 0.000300, loss_epoch: 2.935
Epoch: 110, time(sec): 0.300, lr= 0.000300, loss_epoch: 2.948
Epoch: 120, time(sec): 0.327, lr= 0.000300, loss_epoch: 2.477
Epoch: 130, time(sec): 0.354, lr= 0.000300, 

## Stage 7 : Multiple Attention Heads (MHA)

#### Predict next token t+1 given the context = { tokens ≤ t }

#### Aggregator of tokens is the multi-head attention operator : 

$H \ \leftarrow \ \textrm{Concat}_\textrm{MH} \Big( \textrm{Softmax}( QK^T / \sqrt{d} ) V \Big) W_O $,

with learnable dictionary $Q=W_Q H$ (Query), $K=W_K H$ (Key), and $V=W_V H$ (Value),

and learnable $W_O$ (that linearly combined all single heads),

and initial embedding $H \ \leftarrow \ \textrm{Token-Embedding} + \textrm{PE}$.


In [13]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# single head attention layer
class head_attention(nn.Module):
    def __init__(self, d, d_head, context_length):
        super().__init__()
        self.query = nn.Linear(d, d_head, bias=False) # query embedding layer
        self.key = nn.Linear(d, d_head, bias=False) # key embedding layer
        self.value = nn.Linear(d, d_head) # value embedding layer
        self.mask = torch.tril(torch.ones(context_length, context_length)).long() # mask to use previous tokens only : { token(<=t) }, size=[context_length, context_length]
    def forward(self, H):
        Q = self.query(H) # size=[batch_size, batch_length, d_head]
        K = self.key(H) # size=[batch_size, batch_length, d_head]
        V = self.value(H) # size=[batch_size, batch_length, d_head]
        attention_score = Q @ K.transpose(2,1) * H.size(2)**-0.5 # QK^T/sqrt(d_head), (B,L,d_head) @ (B,d_head,L) => (B,L,L), size=[batch_size, batch_length, batch_length)
        attention_score = attention_score.masked_fill(self.mask==0, value=float('-inf')) # softmax(-inf)=0 prevents using next tokens for prediction, size=(batch_size, batch_len, batch_len)
        attention_score = torch.softmax(attention_score, dim=2) # sum weights = 1, size=[batch_size, batch_length, batch_len]
        H_head = attention_score @ V # softmax( QK^T / sqrt(d_head) ) V, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d_head]
        return H_head

# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads # dim_head = d / num_heads, usually dimension per head is 64
        assert d == d_head * num_heads # check divisibility
        self.MHA = nn.ModuleList([ head_attention(d, d_head, context_length) for _ in range(num_heads) ])
        self.combined_heads = nn.Linear(d, d) # combination layer
    def forward(self, H):
        H_heads = []
        for HA_layer in self.MHA:
            H_heads.append(HA_layer(H)) # size=[batch_size, batch_length, d_head]
        H_heads = torch.cat(H_heads, dim=2) # size=[batch_size, batch_length, d]
        H_heads = self.combined_heads(H_heads) # size=[batch_size, batch_length, d]
        return H_heads

class MHA_LM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.MHA = multiple_head_attention(d, context_length, num_heads) # multiple self-attention layers
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        seq_pos_encoding = torch.arange(batch_seq.size(1)) # positional encoding = {0,1,2,...,batch_length-1}
        H = self.token2vec(batch_seq) + self.PE_embedding(seq_pos_encoding).unsqueeze(0) # size=[batch_size, batch_length, d]
        batch_seq_vec = self.MHA(H) # (single) attention head, size=[batch_size, batch_length, d]
        batch_scores = self.token_prediction(batch_seq_vec) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token

# batching parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 5; batch_length = 20 # bebug
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# network parameters
d = 128 # embedding dimension
num_heads = 16
print('num_tokens: %d, d: %d, batch_length: %d, num_heads: %d\n' % (num_tokens, d, batch_length, num_heads) )
MHA_LLMnet = MHA_LM(num_tokens, d, batch_length, num_heads)
num_param = number_param(MHA_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Train network to predict next token
optimizer = torch.optim.AdamW(MHA_LLMnet.parameters(), lr=3e-4) # standard optimizer for LMs
num_epochs = 201 # 101(debug), number of epochs
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for _ in range(num_batch): # number of batches into one epoch
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
        batch_scores = MHA_LLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    loss_epoch = running_loss / num_batch
    if not epoch%10:
        print('Epoch: %d, time(sec): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, time.time()-start, optimizer.param_groups[0]['lr'], loss_epoch) )


seq_len: 100, batch_size: 5, batch_length: 20, num_subseq: 5, num_batch: 1

num_tokens: 129, d: 128, batch_length: 20, num_heads: 16

num_net_parameters: 101505 / 0.10 million

Epoch: 0, time(sec): 0.017, lr= 0.000300, loss_epoch: 4.884
Epoch: 10, time(sec): 0.173, lr= 0.000300, loss_epoch: 4.701
Epoch: 20, time(sec): 0.330, lr= 0.000300, loss_epoch: 4.543
Epoch: 30, time(sec): 0.486, lr= 0.000300, loss_epoch: 4.276
Epoch: 40, time(sec): 0.644, lr= 0.000300, loss_epoch: 4.000
Epoch: 50, time(sec): 0.801, lr= 0.000300, loss_epoch: 3.764
Epoch: 60, time(sec): 0.959, lr= 0.000300, loss_epoch: 3.570
Epoch: 70, time(sec): 1.115, lr= 0.000300, loss_epoch: 3.643
Epoch: 80, time(sec): 1.272, lr= 0.000300, loss_epoch: 3.305
Epoch: 90, time(sec): 1.429, lr= 0.000300, loss_epoch: 3.139
Epoch: 100, time(sec): 1.585, lr= 0.000300, loss_epoch: 2.979
Epoch: 110, time(sec): 1.742, lr= 0.000300, loss_epoch: 3.061
Epoch: 120, time(sec): 1.898, lr= 0.000300, loss_epoch: 2.555
Epoch: 130, time(sec): 2.055

## Stage 8 : Vanilla Transformer Block = { MHA + Multi-Layer Perceptron (MLP) } + PE

#### Predict next token t+1 given the context = { tokens ≤ t }

#### Aggregator of tokens is the MHA operator followed by a 2-layer Feedforward network : 

$H \ \leftarrow \ \textrm{Concat}_\textrm{MH} \Big( \textrm{Softmax}( QK^T / \sqrt{d} ) V \Big) W_O $,

$H \ \leftarrow \ \textrm{MLP}(H)$

with learnable dictionary $Q=W_Q H$ (Query), $K=W_K H$ (Key), and $V=W_V H$ (Value),

learnable $W_O$ (that linearly combined all single heads),

learnable 2-layer MLP,

and initial embedding $H \ \leftarrow \ \textrm{Token-Embedding} + \textrm{PE}$.


In [14]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# single head attention layer
class head_attention(nn.Module):
    def __init__(self, d, d_head, context_length):
        super().__init__()
        self.query = nn.Linear(d, d_head, bias=False) # query embedding layer
        self.key = nn.Linear(d, d_head, bias=False) # key embedding layer
        self.value = nn.Linear(d, d_head) # value embedding layer
        self.mask = torch.tril(torch.ones(context_length, context_length)).long() # mask to use previous tokens only : { token(<=t) }, size=[context_length, context_length]
    def forward(self, H):
        Q = self.query(H) # size=[batch_size, batch_length, d_head]
        K = self.key(H) # size=[batch_size, batch_length, d_head]
        V = self.value(H) # size=[batch_size, batch_length, d_head]
        attention_score = Q @ K.transpose(2,1) * H.size(2)**-0.5 # QK^T/sqrt(d_head), (B,L,d_head) @ (B,d_head,L) => (B,L,L), size=[batch_size, batch_length, batch_length)
        attention_score = attention_score.masked_fill(self.mask==0, value=float('-inf')) # softmax(-inf)=0 prevents using next tokens for prediction, size=(batch_size, batch_len, batch_len)
        attention_score = torch.softmax(attention_score, dim=2) # sum weights = 1, size=[batch_size, batch_length, batch_len]
        H_head = attention_score @ V # softmax( QK^T / sqrt(d_head) ) V, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d_head]
        return H_head

# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads # dim_head = d / num_heads, usually dimension per head is 64
        assert d == d_head * num_heads # check divisibility
        self.MHA = nn.ModuleList([ head_attention(d, d_head, context_length) for _ in range(num_heads) ])
        self.combined_heads = nn.Linear(d, d) # combination layer
    def forward(self, H):
        H_heads = []
        for HA_layer in self.MHA:
            H_heads.append(HA_layer(H)) # size=[batch_size, batch_length, d_head]
        H_heads = torch.cat(H_heads, dim=2) # size=[batch_size, batch_length, d]
        H_heads = self.combined_heads(H_heads) # size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
    def forward(self, H):
        H = self.MHA(H) # size=[batch_size, batch_length, d]
        H = self.MLP(H) # size=[batch_size, batch_length, d]
        return H

class TB_LM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.TB = TransformerBlock(d, context_length, num_heads) # transformer block layer
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        seq_pos_encoding = torch.arange(batch_seq.size(1)) # positional encoding = {0,1,2,...,batch_length-1}
        H = self.token2vec(batch_seq) + self.PE_embedding(seq_pos_encoding).unsqueeze(0) # size=[batch_size, batch_length, d]
        batch_seq_vec = self.TB(H) # (single) transformer block, size=[batch_size, batch_length, d]
        batch_scores = self.token_prediction(batch_seq_vec) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token

# batching parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 5; batch_length = 20 # bebug
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# network parameters
d = 128 # embedding dimension
num_heads = 16
print('num_tokens: %d, d: %d, batch_length: %d, num_heads: %d\n' % (num_tokens, d, batch_length, num_heads) )
TB_LLMnet = TB_LM(num_tokens, d, batch_length, num_heads)
num_param = number_param(TB_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Train network to predict next token
optimizer = torch.optim.AdamW(TB_LLMnet.parameters(), lr=3e-4) # standard optimizer for LMs
num_epochs = 201 # 101(debug), number of epochs
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for _ in range(num_batch): # number of batches into one epoch
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
        batch_scores = TB_LLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    loss_epoch = running_loss / num_batch
    if not epoch%10:
        print('Epoch: %d, time(sec): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, time.time()-start, optimizer.param_groups[0]['lr'], loss_epoch) )


seq_len: 100, batch_size: 5, batch_length: 20, num_subseq: 5, num_batch: 1

num_tokens: 129, d: 128, batch_length: 20, num_heads: 16

num_net_parameters: 233217 / 0.23 million

Epoch: 0, time(sec): 0.026, lr= 0.000300, loss_epoch: 4.863
Epoch: 10, time(sec): 0.195, lr= 0.000300, loss_epoch: 4.736
Epoch: 20, time(sec): 0.364, lr= 0.000300, loss_epoch: 4.350
Epoch: 30, time(sec): 0.533, lr= 0.000300, loss_epoch: 4.099
Epoch: 40, time(sec): 0.702, lr= 0.000300, loss_epoch: 3.826
Epoch: 50, time(sec): 0.870, lr= 0.000300, loss_epoch: 3.731
Epoch: 60, time(sec): 1.039, lr= 0.000300, loss_epoch: 3.603
Epoch: 70, time(sec): 1.208, lr= 0.000300, loss_epoch: 3.360
Epoch: 80, time(sec): 1.377, lr= 0.000300, loss_epoch: 3.126
Epoch: 90, time(sec): 1.546, lr= 0.000300, loss_epoch: 2.781
Epoch: 100, time(sec): 1.715, lr= 0.000300, loss_epoch: 2.563
Epoch: 110, time(sec): 1.883, lr= 0.000300, loss_epoch: 2.284
Epoch: 120, time(sec): 2.054, lr= 0.000300, loss_epoch: 1.925
Epoch: 130, time(sec): 2.223

## Stage 9 : Improved Transformer Block = { MHSA + MLP + Layer Normalization (LN) } + PE

#### Predict next token t+1 given the context = { tokens ≤ t }

#### Aggregator of tokens is augmented with normalization layer: 

$H \ \leftarrow \ \textrm{LN}(H)$

$H \ \leftarrow \ \textrm{Concat}_\textrm{MH} \Big( \textrm{Softmax}( QK^T / \sqrt{d} ) V \Big) W_O $,

$H \ \leftarrow \ \textrm{LN}(H)$

$H \ \leftarrow \ \textrm{MLP}(H)$

with learnable dictionary $Q=W_Q H$ (Query), $K=W_K H$ (Key), and $V=W_V H$ (Value),

learnable $W_O$ (that linearly combined all single heads),

learnable 2-layer MLP,

and initial embedding $H \ \leftarrow \ \textrm{Token-Embedding} + \textrm{PE}$.


In [15]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# single head attention layer
class head_attention(nn.Module):
    def __init__(self, d, d_head, context_length):
        super().__init__()
        self.query = nn.Linear(d, d_head, bias=False) # query embedding layer
        self.key = nn.Linear(d, d_head, bias=False) # key embedding layer
        self.value = nn.Linear(d, d_head) # value embedding layer
        self.mask = torch.tril(torch.ones(context_length, context_length)).long() # mask to use previous tokens only : { token(<=t) }, size=[context_length, context_length]
    def forward(self, H):
        Q = self.query(H) # size=[batch_size, batch_length, d_head]
        K = self.key(H) # size=[batch_size, batch_length, d_head]
        V = self.value(H) # size=[batch_size, batch_length, d_head]
        attention_score = Q @ K.transpose(2,1) * H.size(2)**-0.5 # QK^T/sqrt(d_head), (B,L,d_head) @ (B,d_head,L) => (B,L,L), size=[batch_size, batch_length, batch_length)
        attention_score = attention_score.masked_fill(self.mask==0, value=float('-inf')) # softmax(-inf)=0 prevents using next tokens for prediction, size=(batch_size, batch_len, batch_len)
        attention_score = torch.softmax(attention_score, dim=2) # sum weights = 1, size=[batch_size, batch_length, batch_len]
        H_head = attention_score @ V # softmax( QK^T / sqrt(d_head) ) V, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d_head]
        return H_head

# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads # dim_head = d / num_heads, usually dimension per head is 64
        assert d == d_head * num_heads # check divisibility
        self.MHA = nn.ModuleList([ head_attention(d, d_head, context_length) for _ in range(num_heads) ])
        self.combined_heads = nn.Linear(d, d) # combination layer
    def forward(self, H):
        H_heads = []
        for HA_layer in self.MHA:
            H_heads.append(HA_layer(H)) # size=[batch_size, batch_length, d_head]
        H_heads = torch.cat(H_heads, dim=2) # size=[batch_size, batch_length, d]
        H_heads = self.combined_heads(H_heads) # size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H

class TB_LM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.TB = TransformerBlock(d, context_length, num_heads) # transformer block layer
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        seq_pos_encoding = torch.arange(batch_seq.size(1)) # positional encoding = {0,1,2,...,batch_length-1}
        H = self.token2vec(batch_seq) + self.PE_embedding(seq_pos_encoding).unsqueeze(0) # size=[batch_size, batch_length, d]
        batch_seq_vec = self.TB(H) # (single) transformer block, size=[batch_size, batch_length, d]
        batch_scores = self.token_prediction(batch_seq_vec) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token

# batching parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 5; batch_length = 20 # bebug
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# network parameters
d = 128 # embedding dimension
num_heads = 16
print('num_tokens: %d, d: %d, batch_length: %d, num_heads: %d\n' % (num_tokens, d, batch_length, num_heads) )
TB_LLMnet = TB_LM(num_tokens, d, batch_length, num_heads)
num_param = number_param(TB_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Train network to predict next token
optimizer = torch.optim.AdamW(TB_LLMnet.parameters(), lr=3e-4) # standard optimizer for LMs
num_epochs = 201 # 101(debug), number of epochs
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for _ in range(num_batch): # number of batches into one epoch
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
        batch_scores = TB_LLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    loss_epoch = running_loss / num_batch
    if not epoch%10:
        print('Epoch: %d, time(sec): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, time.time()-start, optimizer.param_groups[0]['lr'], loss_epoch) )


seq_len: 100, batch_size: 5, batch_length: 20, num_subseq: 5, num_batch: 1

num_tokens: 129, d: 128, batch_length: 20, num_heads: 16

num_net_parameters: 233729 / 0.23 million

Epoch: 0, time(sec): 0.019, lr= 0.000300, loss_epoch: 4.896
Epoch: 10, time(sec): 0.192, lr= 0.000300, loss_epoch: 4.382
Epoch: 20, time(sec): 0.365, lr= 0.000300, loss_epoch: 3.999
Epoch: 30, time(sec): 0.538, lr= 0.000300, loss_epoch: 3.654
Epoch: 40, time(sec): 0.712, lr= 0.000300, loss_epoch: 3.259
Epoch: 50, time(sec): 0.885, lr= 0.000300, loss_epoch: 2.742
Epoch: 60, time(sec): 1.058, lr= 0.000300, loss_epoch: 2.372
Epoch: 70, time(sec): 1.231, lr= 0.000300, loss_epoch: 2.019
Epoch: 80, time(sec): 1.404, lr= 0.000300, loss_epoch: 2.008
Epoch: 90, time(sec): 1.579, lr= 0.000300, loss_epoch: 1.652
Epoch: 100, time(sec): 1.752, lr= 0.000300, loss_epoch: 1.365
Epoch: 110, time(sec): 1.925, lr= 0.000300, loss_epoch: 1.112
Epoch: 120, time(sec): 2.098, lr= 0.000300, loss_epoch: 0.775
Epoch: 130, time(sec): 2.271

## Stage 10 : Standard Transformer Block = { MHSA + MLP + LN + Residual Connection (RC) } + PE

#### Predict next token t+1 given the context = { tokens ≤ t }

#### Aggregator of tokens is augmented with residual connection: 

$H \ \leftarrow \ H + \textrm{Concat}_\textrm{MH} \Big( \textrm{Softmax}( QK^T / \sqrt{d} ) V \Big) W_O $,

$H \ \leftarrow \ H + \textrm{MLP}(\textrm{LN}(H))$

with learnable dictionary $Q=W_Q \textrm{LN}(H)$ (Query), $K=W_K \textrm{LN}(H)$ (Key), and $V=W_V \textrm{LN}(H)$ (Value),

learnable $W_O$ (that linearly combined all single heads),

learnable 2-layer MLP,

and initial embedding $H \ \leftarrow \ \textrm{Token-Embedding} + \textrm{PE}$.


In [16]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# single head attention layer
class head_attention(nn.Module):
    def __init__(self, d, d_head, context_length):
        super().__init__()
        self.query = nn.Linear(d, d_head, bias=False) # query embedding layer
        self.key = nn.Linear(d, d_head, bias=False) # key embedding layer
        self.value = nn.Linear(d, d_head) # value embedding layer
        self.mask = torch.tril(torch.ones(context_length, context_length)).long() # mask to use previous tokens only : { token(<=t) }, size=[context_length, context_length]
    def forward(self, H):
        Q = self.query(H) # size=[batch_size, batch_length, d_head]
        K = self.key(H) # size=[batch_size, batch_length, d_head]
        V = self.value(H) # size=[batch_size, batch_length, d_head]
        attention_score = Q @ K.transpose(2,1) * H.size(2)**-0.5 # QK^T/sqrt(d_head), (B,L,d_head) @ (B,d_head,L) => (B,L,L), size=[batch_size, batch_length, batch_length)
        attention_score = attention_score.masked_fill(self.mask==0, value=float('-inf')) # softmax(-inf)=0 prevents using next tokens for prediction, size=(batch_size, batch_len, batch_len)
        attention_score = torch.softmax(attention_score, dim=2) # sum weights = 1, size=[batch_size, batch_length, batch_len]
        H_head = attention_score @ V # softmax( QK^T / sqrt(d_head) ) V, (B,L,L) @ (B,L,d) => (B,L,d), size=[batch_size, batch_length, d_head]
        return H_head

# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads # dim_head = d / num_heads, usually dimension per head is 64
        assert d == d_head * num_heads # check divisibility
        self.MHA = nn.ModuleList([ head_attention(d, d_head, context_length) for _ in range(num_heads) ])
        self.combined_heads = nn.Linear(d, d) # combination layer
    def forward(self, H):
        H_heads = []
        for HA_layer in self.MHA:
            H_heads.append(HA_layer(H)) # size=[batch_size, batch_length, d_head]
        H_heads = torch.cat(H_heads, dim=2) # size=[batch_size, batch_length, d]
        H_heads = self.combined_heads(H_heads) # size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H

class TB_LM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.TB = TransformerBlock(d, context_length, num_heads) # transformer block layer
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        seq_pos_encoding = torch.arange(batch_seq.size(1)) # positional encoding = {0,1,2,...,batch_length-1}
        H = self.token2vec(batch_seq) + self.PE_embedding(seq_pos_encoding).unsqueeze(0) # size=[batch_size, batch_length, d]
        batch_seq_vec = self.TB(H) # (single) transformer block, size=[batch_size, batch_length, d]
        batch_scores = self.token_prediction(batch_seq_vec) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token

# batching parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 5; batch_length = 20 # bebug
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# network parameters
d = 128 # embedding dimension
num_heads = 16
dropout = 0.1
print('num_tokens: %d, d: %d, batch_length: %d, num_heads: %d\n' % (num_tokens, d, batch_length, num_heads) )
TB_LLMnet = TB_LM(num_tokens, d, batch_length, num_heads)
num_param = number_param(TB_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Train network to predict next token
optimizer = torch.optim.AdamW(TB_LLMnet.parameters(), lr=3e-4) # standard optimizer for LMs
num_epochs = 201 # 101(debug), number of epochs
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for _ in range(num_batch): # number of batches into one epoch
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
        batch_scores = TB_LLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    loss_epoch = running_loss / num_batch
    if not epoch%10:
        print('Epoch: %d, time(sec): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, time.time()-start, optimizer.param_groups[0]['lr'], loss_epoch) )


seq_len: 100, batch_size: 5, batch_length: 20, num_subseq: 5, num_batch: 1

num_tokens: 129, d: 128, batch_length: 20, num_heads: 16

num_net_parameters: 233729 / 0.23 million

Epoch: 0, time(sec): 0.019, lr= 0.000300, loss_epoch: 5.099
Epoch: 10, time(sec): 0.193, lr= 0.000300, loss_epoch: 4.482
Epoch: 20, time(sec): 0.367, lr= 0.000300, loss_epoch: 3.897
Epoch: 30, time(sec): 0.542, lr= 0.000300, loss_epoch: 3.421
Epoch: 40, time(sec): 0.715, lr= 0.000300, loss_epoch: 3.078
Epoch: 50, time(sec): 0.889, lr= 0.000300, loss_epoch: 2.665
Epoch: 60, time(sec): 1.063, lr= 0.000300, loss_epoch: 2.131
Epoch: 70, time(sec): 1.236, lr= 0.000300, loss_epoch: 1.832
Epoch: 80, time(sec): 1.410, lr= 0.000300, loss_epoch: 1.711
Epoch: 90, time(sec): 1.584, lr= 0.000300, loss_epoch: 1.405
Epoch: 100, time(sec): 1.758, lr= 0.000300, loss_epoch: 1.016
Epoch: 110, time(sec): 1.931, lr= 0.000300, loss_epoch: 0.925
Epoch: 120, time(sec): 2.108, lr= 0.000300, loss_epoch: 0.755
Epoch: 130, time(sec): 2.281

## Stage 11 : PyTorch implementation of Multi-Head Attention (MHA) vs. my implementation

#### Predict next token t+1 given the context = { tokens ≤ t }

#### Aggregator of tokens is augmented with residual connection: 

$H \ \leftarrow \ H + \textrm{MHA}_\textrm{PyTorch}(\textrm{LN}(H))$,

$H \ \leftarrow \ H + \textrm{MLP}(\textrm{LN}(H))$

and initial embedding $H \ \leftarrow \ \textrm{Token-Embedding} + \textrm{PE}$.


In [17]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec
        
# multiple attention heads layer -- PyTorch implementation
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads
        assert d == d_head * num_heads # check divisiblity
        self.MHA = nn.MultiheadAttention(d, num_heads, batch_first=True)
        self.mask = torch.tril(torch.ones(context_length, context_length))==0 # mask to make attention to previous tokens only : { token(<=t) }, size=(context_length,context_length)
                   # torch.tril(ones) = True in the up-right part, True means *no* attention allowed in pytorch implementation
    def forward(self, H):
        H_heads = self.MHA(H, H, H, attn_mask=self.mask)[0] # size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H

class TBpytorch_LM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.TB = TransformerBlock(d, context_length, num_heads) # transformer block layer
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        seq_pos_encoding = torch.arange(batch_seq.size(1)) # positional encoding = {0,1,2,...,batch_length-1}
        H = self.token2vec(batch_seq) + self.PE_embedding(seq_pos_encoding).unsqueeze(0) # size=[batch_size, batch_length, d]
        batch_seq_vec = self.TB(H) # (single) transformer block, size=[batch_size, batch_length, d]
        batch_scores = self.token_prediction(batch_seq_vec) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token

# batching parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 5; batch_length = 20 # bebug
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# network parameters
d = 128 # embedding dimension
num_heads = 16
print('num_tokens: %d, d: %d, batch_length: %d, num_heads: %d\n' % (num_tokens, d, batch_length, num_heads) )
TB_LLMnet = TBpytorch_LM(num_tokens, d, batch_length, num_heads)
num_param = number_param(TB_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Train network to predict next token
optimizer = torch.optim.AdamW(TB_LLMnet.parameters(), lr=3e-4) # standard optimizer for LMs
num_epochs = 201 # 101(debug), number of epochs
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for _ in range(num_batch): # number of batches into one epoch
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
        batch_scores = TB_LLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    loss_epoch = running_loss / num_batch
    if not epoch%10:
        print('Epoch: %d, time(sec): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, time.time()-start, optimizer.param_groups[0]['lr'], loss_epoch) )

# my implementation      : Time(sec): 17.707 / loss_epoch: 1.387  => 3x slower
# pytorch implementation : Time(sec): 6.556  / loss_epoch: 0.940


seq_len: 100, batch_size: 5, batch_length: 20, num_subseq: 5, num_batch: 1

num_tokens: 129, d: 128, batch_length: 20, num_heads: 16

num_net_parameters: 233985 / 0.23 million

Epoch: 0, time(sec): 0.007, lr= 0.000300, loss_epoch: 5.323
Epoch: 10, time(sec): 0.056, lr= 0.000300, loss_epoch: 4.621
Epoch: 20, time(sec): 0.104, lr= 0.000300, loss_epoch: 4.019
Epoch: 30, time(sec): 0.152, lr= 0.000300, loss_epoch: 3.557
Epoch: 40, time(sec): 0.201, lr= 0.000300, loss_epoch: 3.066
Epoch: 50, time(sec): 0.249, lr= 0.000300, loss_epoch: 2.480
Epoch: 60, time(sec): 0.298, lr= 0.000300, loss_epoch: 2.193
Epoch: 70, time(sec): 0.346, lr= 0.000300, loss_epoch: 1.526
Epoch: 80, time(sec): 0.394, lr= 0.000300, loss_epoch: 1.351
Epoch: 90, time(sec): 0.442, lr= 0.000300, loss_epoch: 1.264
Epoch: 100, time(sec): 0.492, lr= 0.000300, loss_epoch: 0.876
Epoch: 110, time(sec): 0.541, lr= 0.000300, loss_epoch: 0.692
Epoch: 120, time(sec): 0.589, lr= 0.000300, loss_epoch: 0.539
Epoch: 130, time(sec): 0.637

## Stage 12 : Multiple Transformer Blocks

#### Predict next token t+1 given the context = { tokens ≤ t }

#### Aggregator of tokens is augmented with residual connection:

For layer = 1 to L :

- $H \ \leftarrow \ H + \textrm{MHA}_\textrm{PyTorch}(\textrm{LN}(H))$,

- $H \ \leftarrow \ H + \textrm{MLP}(\textrm{LN}(H))$

with initial embedding $H \ \leftarrow \ \textrm{Token-Embedding} + \textrm{PE}$.


In [18]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads
        assert d == d_head * num_heads # check divisiblity
        self.MHA = nn.MultiheadAttention(d, num_heads, batch_first=True)
        self.mask = torch.tril(torch.ones(context_length, context_length))==0 # mask to make attention to previous tokens only : { token(<=t) }, size=(context_length,context_length)
                   # torch.tril(ones) = True in the up-right part, True means *no* attention allowed in pytorch implementation
    def forward(self, H):
        H_heads = self.MHA(H, H, H, attn_mask=self.mask)[0] # size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H

class MTB_LM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads, num_layers):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.transformer_blocks = nn.ModuleList([ TransformerBlock(d, context_length, num_heads) for _ in range(num_layers) ]) # multiple transformer block layers
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        seq_pos_encoding = torch.arange(batch_seq.size(1)) # positional encoding = {0,1,2,...,batch_length-1}
        H = self.token2vec(batch_seq) + self.PE_embedding(seq_pos_encoding).unsqueeze(0) # size=[batch_size, batch_length, d]
        for transformer_block in self.transformer_blocks:
            H = transformer_block(H) # size=[batch_size, batch_length, d]
        batch_scores = self.token_prediction(H) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token

# batching parameters
seq_len = seq.size(0) # length of the long sequence
batch_size = 5; batch_length = 20 # bebug
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# network parameters
d = 128 # embedding dimension
num_heads = 16
num_layers = 2
print('num_tokens: %d, d: %d, batch_length: %d, num_heads: %d, num_layers: %d\n' % (num_tokens, d, batch_length, num_heads, num_layers) )
MTB_LLMnet = MTB_LM(num_tokens, d, batch_length, num_heads, num_layers)
num_param = number_param(MTB_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# Train network to predict next token
optimizer = torch.optim.AdamW(MTB_LLMnet.parameters(), lr=3e-4) # standard optimizer for LMs
num_epochs = 201 # 101(debug), number of epochs
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for _ in range(num_batch): # number of batches into one epoch
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx) # generate a batch of subsequences
        batch_scores = MTB_LLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    loss_epoch = running_loss / num_batch
    if not epoch%10:
        print('Epoch: %d, time(sec): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, time.time()-start, optimizer.param_groups[0]['lr'], loss_epoch) )


seq_len: 100, batch_size: 5, batch_length: 20, num_subseq: 5, num_batch: 1

num_tokens: 129, d: 128, batch_length: 20, num_heads: 16, num_layers: 2

num_net_parameters: 432257 / 0.43 million

Epoch: 0, time(sec): 0.010, lr= 0.000300, loss_epoch: 5.220
Epoch: 10, time(sec): 0.093, lr= 0.000300, loss_epoch: 4.259
Epoch: 20, time(sec): 0.175, lr= 0.000300, loss_epoch: 3.523
Epoch: 30, time(sec): 0.258, lr= 0.000300, loss_epoch: 2.808
Epoch: 40, time(sec): 0.339, lr= 0.000300, loss_epoch: 2.252
Epoch: 50, time(sec): 0.421, lr= 0.000300, loss_epoch: 1.759
Epoch: 60, time(sec): 0.504, lr= 0.000300, loss_epoch: 1.293
Epoch: 70, time(sec): 0.587, lr= 0.000300, loss_epoch: 1.037
Epoch: 80, time(sec): 0.669, lr= 0.000300, loss_epoch: 0.753
Epoch: 90, time(sec): 0.752, lr= 0.000300, loss_epoch: 0.581
Epoch: 100, time(sec): 0.834, lr= 0.000300, loss_epoch: 0.361
Epoch: 110, time(sec): 0.915, lr= 0.000300, loss_epoch: 0.447
Epoch: 120, time(sec): 0.998, lr= 0.000300, loss_epoch: 0.571
Epoch: 130, t

## Final stage : GPU training, warm-up learning rate, save final network

## Output : Self-Supervised Learning (SSL) of LLM (Step 1 of LLM training)


In [19]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# GPU training
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    device = torch.device("cuda") # use GPU
else:
    device = torch.device("cpu")
print('device:',device,'\n')

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads
        assert d == d_head * num_heads # check divisiblity
        self.MHA = nn.MultiheadAttention(d, num_heads, batch_first=True)
        self.mask = torch.tril(torch.ones(context_length, context_length)).to(device)==0 # mask to make attention to previous tokens only : { token(<=t) }, size=(context_length,context_length)
                   # torch.tril(ones) = True in the up-right part, True means *no* attention allowed in pytorch implementation
        self.context_length = context_length
    def forward(self, H):
        if H.size(1) == self.context_length: # training <==
            attn_mask = self.mask
        else: # when batch_length not= context_length, e.g. inference time / sequence generation <==
            current_batch_length = H.size(1)
            attn_mask = torch.tril(torch.ones(current_batch_length, current_batch_length)).to(device)==0
        H_heads = self.MHA(H, H, H, attn_mask=attn_mask.to(device))[0] # pytorch implementation, size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H

# class of self-supervised learning LLM network (Step 1)
class SSL_LLM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads, num_layers):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.transformer_blocks = nn.ModuleList([ TransformerBlock(d, context_length, num_heads) for _ in range(num_layers) ]) # multiple transformer block layers
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        seq_pos_encoding = torch.arange(batch_seq.size(1)).to(device) # positional encoding = {0,1,2,...,batch_length-1}
        H = self.token2vec(batch_seq) + self.PE_embedding(seq_pos_encoding).unsqueeze(0) # size=[batch_size, batch_length, d]
        for transformer_block in self.transformer_blocks:
            H = transformer_block(H) # size=[batch_size, batch_length, d]
        batch_scores = self.token_prediction(H) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token

# generate a new sentence of any length
def generate(LLMnet, prompt_seq, max_length_gen_seq):
    LLMnet.eval()
    with torch.no_grad():
        gen_seq = prompt_seq # an initial sequence (a.k.a. prompt) to generate a longer sequence with LLMnet
        for k in range(max_length_gen_seq):
            context_seq = gen_seq[:,-batch_length:] # size=[1, <=batch_length]
            score = LLMnet(context_seq) # size=[1, batch_length, num_tokens]
            score_last_token = score[:,-1,:].squeeze(dim=1) # size=[1, num_tokens]
            prob_last_token = torch.softmax(score_last_token, dim=1) # size=[1, num_tokens]
            #idx_next_token = torch.multinomial(prob_last_token, num_samples=1) # size=[1,1]
            idx_next_token = torch.max(prob_last_token, dim=1).indices[0].view(1,1) # size=[1,1]
            gen_seq = torch.cat((gen_seq, idx_next_token), dim=1) # append next token, (size=[1, num_tokens+1]
    LLMnet.train()
    return gen_seq


# network parameters
d_head = 32; num_heads = 4; d = num_heads * d_head; num_layers = 2; batch_size = 2; batch_length = 20 # bebug
# d_head = 64; num_heads = 6; d = num_heads * d_head; num_layers = 6; batch_size = 500; batch_length = 40 # 10M 
print('num_tokens: %d, d: %d, batch_length: %d, num_heads: %d, num_layers: %d\n' % (num_tokens, d, batch_length, num_heads, num_layers) )
SSL_LLMnet = SSL_LLM(num_tokens, d, batch_length, num_heads, num_layers)
SSL_LLMnet = SSL_LLMnet.to(device)
num_param = number_param(SSL_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )

# save checkpoint
net_parameters = {}
net_parameters['num_tokens'] = num_tokens
net_parameters['d'] = d
net_parameters['num_heads'] = num_heads
net_parameters['batch_length'] = batch_length
net_parameters['num_layers'] = num_layers
checkpoint_dir = os.path.join("checkpoint")
import os
os.makedirs(checkpoint_dir, exist_ok=True)
print('checkpoint file :', checkpoint_dir + '/step1_checkpoint_SSL_LLM_' + time_stamp + '.pkl', '\n')

# batching parameters
seq_len = seq.size(0) # length of the long sequence
num_subseq = seq_len // batch_length # number of subsequences
num_batch = seq_len // (batch_size * batch_length) # number of batches
start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # new starting index at each new epoch, random integer in {0,batch_length-1}
list_batch_idx = torch.arange(num_batch) # list of batch indices, [0,1,...,num_batch-1]
print('seq_len: %d, batch_size: %d, batch_length: %d, num_subseq: %d, num_batch: %d\n' % (seq_len, batch_size, batch_length, num_subseq, num_batch) )

# optimizer
num_epochs = 100 # debug
# num_epochs = 1 # number of epochs
print('num_epochs :', num_epochs)
init_lr = 3e-4
final_lr = 1e-6
warmup_steps = 100
total_steps = num_batch * num_epochs  # Total training steps (e.g., 1 epoch)
optimizer = torch.optim.AdamW(SSL_LLMnet.parameters(), lr=init_lr)
def lr_lambda(current_step):
    # 1. Linear Warmup Phase
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    # 2. Cosine Annealing Phase
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    # Clamp progress at 1.0 to avoid errors if training goes longer than expected
    progress = min(1.0, progress)
    cosine_decay = 0.5 * (1.0 + torch.cos(torch.tensor(torch.pi * progress)).item()) # Standard cosine
    # Calculate the ratio between final_lr and init_lr to ensure we hit the floor
    lr_ratio = final_lr / init_lr
    # The multiplier starts at 1.0 and ends at lr_ratio
    return lr_ratio + (1.0 - lr_ratio) * cosine_decay
# Scheduler
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx, device='cpu'):
    # input  : list_batch_idx = [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14], size=[15] 
    # output : list_batch_idx = [ 8,  2,  0,  6,  1, 10, 14,  9, 13,  7,  4, 11], size=[12], with e.g. batch_size=3
    num_sub_sequences = list_batch_idx.size(0)
    if num_sub_sequences == 0: # no more sequences
        return torch.tensor([]), torch.tensor([]), torch.tensor([], dtype=torch.long)
    bs = min(batch_size, num_sub_sequences) # if fewer samples remain than batch_size
    # shuffle indices and split 
    perm = torch.randperm(num_sub_sequences)
    selected_indices = perm[:bs]
    remaining_indices = perm[bs:]
    batch_idx = list_batch_idx[selected_indices] # actual batch indices
    new_list_batch_idx = list_batch_idx[remaining_indices] # remaining pool
    # Calculate the starting position for every sequence in the batch at once
    chunk_starts = start_idx + batch_idx * batch_length # size=[batch_size], e.g. chunk_starts=[20, 74, 32]
    offsets = torch.arange(batch_length).unsqueeze(0) # [0, 1, 2, ..., batch_length-1], size=[batch_length]
    # Broadcast to create a full 2D index matrix
    input_indices = chunk_starts.unsqueeze(1) + offsets # input_indices[i,j]=chunk_starts[i]+j, size=[batch_size, batch_length]
    # input_indices = [ [20, 21, 22, 23, 24, 25], [74, 75, 76, 77, 78, 79], [32, 33, 34, 35, 36, 37] ]
    target_indices = input_indices + 1 # targets are just input_indices shifted by +1
    # target_indices = [ [21, 22, 23, 24, 25, 26], [75, 76, 77, 78, 79, 80], [33, 34, 35, 36, 37, 38] ]
    batch_seq = seq[input_indices]
    # batch_seq = [ [17, 18,  5, 19, 20,  5], [15,  5, 51, 23, 52, 19], [ 5, 25, 17, 26, 27, 18] ]
    target_seq = seq[target_indices]
    # target_seq = [ [18,  5, 19, 20,  5, 21], [ 5, 51, 23, 52, 19, 53], [25, 17, 26, 27, 18,  5] ]
    return batch_seq, target_seq, new_list_batch_idx


# Train network to predict next token
start = time.time()
for epoch in range(num_epochs): # number of epochs
    list_batch_idx = torch.arange(num_subseq-1) # list of batch indices
    start_idx = torch.randint(low=0, high=batch_length, size=(1,)) # size=[1]
    running_loss = 0.0 # tracking total loss value
    for idx_batch in range(num_batch): # number of batches into one epoch
        if not idx_batch%100 and idx_batch>0:
            print(f'idx_batch : {idx_batch}, time(min) : {(time.time()-start)/60:.3f}')
        batch_seq, target_seq, list_batch_idx = get_batch(seq, batch_size, batch_length, start_idx, list_batch_idx, device) # generate a batch of subsequences
        batch_seq, target_seq = batch_seq.to(device), target_seq.to(device) 
        batch_scores = SSL_LLMnet(batch_seq) # size=[batch_size, batch_length, num_tokens]
        loss = nn.CrossEntropyLoss()(batch_scores.view(batch_scores.size(0)*batch_length, num_tokens), target_seq.view(batch_scores.size(0)*batch_length)) # classification loss over dict of tokens
        running_loss += loss.detach().cpu().item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step() # warmup scheduler
    loss_epoch = running_loss / num_batch
    if not epoch%1: 
        print('Epoch: %d, time(min): %.3f, lr= %.6f, loss_epoch: %.3f' % (epoch, (time.time()-start)/60, optimizer.param_groups[0]['lr'], loss_epoch) )
         # save checkpoint
        torch.save({
            'epoch': epoch,
            'tot_time': time.time()-start,
            'loss': loss_epoch,
            'net_parameters': net_parameters,
            'SSL_LLMnet_dict': SSL_LLMnet.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            }, '{}.pkl'.format(checkpoint_dir + "/step1_checkpoint_SSL_LLM_" + time_stamp ))
        # check prediction performance
        prompt_seq = get_batch(seq, batch_size=1, batch_length=4, start_idx=10, list_batch_idx=torch.arange(num_subseq-1))[0].to(device) # generate a small sequence to complete, size=[1,batch_length]
        prompt_tokens = func_tokens2str(func_indices2tokens(prompt_seq[0].tolist()))
        print('sequence   :', prompt_tokens)
        gen_seq = generate(SSL_LLMnet, prompt_seq, max_length_gen_seq=batch_length)[0][prompt_seq[0].size(0):]
        seq_tokens = func_tokens2str(func_indices2tokens(gen_seq.tolist()))
        print('prediction :             ', seq_tokens,'\n')
    # Stopping condition
    if loss_epoch < 0.1:
        print("\n loss value is small -- training stopped\n")
        break

# GPU training time : Epoch: 0, time(min): 37.975, lr= 0.000001, loss_epoch: 1.044


NVIDIA RTX A5000
device: cuda 

num_tokens: 129, d: 128, batch_length: 20, num_heads: 4, num_layers: 2

num_net_parameters: 432257 / 0.43 million

checkpoint file : checkpoint/step1_checkpoint_SSL_LLM_26-02-07--18-48-32.pkl 

seq_len: 100, batch_size: 2, batch_length: 20, num_subseq: 5, num_batch: 2

num_epochs : 100
Epoch: 0, time(min): 0.014, lr= 0.000006, loss_epoch: 5.161
sequence   : 5 6 7 8
prediction :              23 20 13 28 87 55 Let 96 77 26 96 a 54 33 7 86 2 52 51 <PAD> 

Epoch: 1, time(min): 0.015, lr= 0.000012, loss_epoch: 5.237
sequence   : <SEP> 91 81 1
prediction :              23 20 13 28 87 55 Let 96 77 26 96 a 54 33 7 86 2 52 51 <PAD> 

Epoch: 2, time(min): 0.015, lr= 0.000018, loss_epoch: 5.217
sequence   : 2 3 4 75
prediction :              at 22 starts 28 87 55 91 88 53 type 71 23 27 3 32 26 96 19 40 69 

Epoch: 3, time(min): 0.016, lr= 0.000024, loss_epoch: 5.062
sequence   : <SEP> 91 81 1
prediction :              23 20 13 28 87 55 Let 96 77 26 96 a 54 33 7 86 

## Loading the pre-trained SSL-LLM model and generate

In [1]:
import torch
import torch.nn as nn
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    device = torch.device("cuda") # use GPU
else:
    device = torch.device("cpu")
print('device:',device)

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param
    
# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec
# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads
        assert d == d_head * num_heads # check divisiblity
        self.MHA = nn.MultiheadAttention(d, num_heads, batch_first=True)
        self.mask = torch.tril(torch.ones(context_length, context_length)).to(device)==0 # mask to make attention to previous tokens only : { token(<=t) }, size=(context_length,context_length)
                   # torch.tril(ones) = True in the up-right part, True means *no* attention allowed in pytorch implementation
        self.context_length = context_length
    def forward(self, H):
        if H.size(1) == self.context_length: # training <==
            attn_mask = self.mask
        else: # when batch_length not= context_length, e.g. inference time / sequence generation <==
            current_batch_length = H.size(1)
            attn_mask = torch.tril(torch.ones(current_batch_length, current_batch_length)).to(device)==0
        H_heads = self.MHA(H, H, H, attn_mask=attn_mask.to(device))[0] # pytorch implementation, size=[batch_size, batch_length, d]
        return H_heads
# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H    
# class of self-supervised learning LLM network (Step 1)
class SSL_LLM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads, num_layers):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.transformer_blocks = nn.ModuleList([ TransformerBlock(d, context_length, num_heads) for _ in range(num_layers) ]) # multiple transformer block layers
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
    def forward(self, batch_seq):
        seq_pos_encoding = torch.arange(batch_seq.size(1)).to(device) # positional encoding = {0,1,2,...,batch_length-1}
        H = self.token2vec(batch_seq) + self.PE_embedding(seq_pos_encoding).unsqueeze(0) # size=[batch_size, batch_length, d]
        for transformer_block in self.transformer_blocks:
            H = transformer_block(H) # size=[batch_size, batch_length, d]
        batch_scores = self.token_prediction(H) # size=[batch_size, batch_length, num_tokens]
        return batch_scores # return prediction scores for next token        

# Download pre-trained SSL-LLM net
import os
checkpoint_file = 'checkpoint/step1_checkpoint_SSL_LLM_26-02-07--18-48-32_200M_10M.pkl' 
print('checkpoint_file:', checkpoint_file, '\n')
if not os.path.exists(checkpoint_file):
    print(f'Downloading {checkpoint_file} ...')
    dropbox_url = "https://www.dropbox.com/scl/fi/dlmsnuk43z9q204ptjtvn/step1_checkpoint_SSL_LLM_26-02-07-18-48-32_200M_10M.pkl?rlkey=owy7mky13kou2i04m60l6jg4c&dl=1"
    !wget -q "{dropbox_url}" -O "{checkpoint_file}"
checkpoint = torch.load(checkpoint_file, map_location=device)
net_parameters = checkpoint['net_parameters']
num_tokens = net_parameters['num_tokens']
d = net_parameters['d']
num_heads = net_parameters['num_heads']
batch_length = net_parameters['batch_length']
num_layers = net_parameters['num_layers']
epoch = checkpoint['epoch']
tot_time = checkpoint['tot_time']
loss = checkpoint['loss']
print('Load pre-trained SSL-LLM: \n checkpoint file: {:s}\n epoch: {:d}, time: {:.3f}min, loss={:.4f}'.format(checkpoint_file,epoch,tot_time,loss))
print(' num_tokens: %d, d: %d, batch_length: %d, num_heads: %d, num_layers: %d\n' % (num_tokens, d, batch_length, num_heads, num_layers) )
SSL_LLMnet = SSL_LLM(num_tokens, d, batch_length, num_heads, num_layers)
SSL_LLMnet = SSL_LLMnet.to(device)
SSL_LLMnet.load_state_dict(checkpoint['SSL_LLMnet_dict'])
num_param = number_param(SSL_LLMnet)
print('num_net_parameters: %d / %.2f million\n' % (num_param, num_param/1e6) )
del checkpoint

file_dictionary = 'dataset/step1_SSL_dictionary_26-02-07--18-48-32_200M.pt' # GPU pre-trained
print('file_dictionary:', file_dictionary, '\n')
import os
os.makedirs('dataset', exist_ok=True)
if not os.path.exists(file_dictionary):
    print(f'Downloading {file_dictionary} ...')
    dropbox_url = "https://www.dropbox.com/scl/fi/jvb5xkk0kfzwrm7vzha3o/step1_SSL_dictionary_26-02-07-18-48-32_200M.pt?rlkey=tprax7nqqpvogo04fhe7m2okd&dl=1"
    !wget -q "{dropbox_url}" -O "{file_dictionary}"
dictionary, num_tokens, token2index, index2token = torch.load(file_dictionary, map_location=device) # load dictionary of tokens
print('dictionary:',dictionary,'\n')
print('num_tokens (unique):',num_tokens,'\n')
print('token2index:', token2index,'\n')
print('index2token:', index2token,'\n')
func_tokens2indices = lambda list_tokens: [token2index[token] for token in list_tokens] # ['Let', '5', 'be', 'the'] => [113, 46, 114, 115]
func_indices2tokens = lambda list_ints: [index2token[integer] for integer in list_ints] # [113, 46, 114, 115] => ['Let', '5', 'be', 'the']
func_str2tokens = lambda input_str: [token_str for token_str in input_str.split()]      # 'Let 5 be the' => ['Let', '5', 'be', 'the']
func_tokens2str = lambda list_str: ' '.join(list_str)                                   # ['Let', '5', 'be', 'the'] => 'Let 5 be the'


device: cpu
checkpoint_file: checkpoint/step1_checkpoint_SSL_LLM_26-02-07--18-48-32_200M_10M.pkl 

Load pre-trained SSL-LLM: 
 checkpoint file: checkpoint/step1_checkpoint_SSL_LLM_26-02-07--18-48-32_200M_10M.pkl
 epoch: 0, time: 2278.492min, loss=1.0443
 num_tokens: 129, d: 384, batch_length: 40, num_heads: 6, num_layers: 6

num_net_parameters: 10761345 / 10.76 million

file_dictionary: dataset/step1_SSL_dictionary_26-02-07--18-48-32_200M.pt 

dictionary: ['51', '57', '63', '69', '75', '81', '87', '93', '99', '<SEP>', '91', '1', '2', '3', '4', '5', '6', '7', '8', '9', '77', '79', '83', '85', '89', '48', '55', '62', '76', '90', '97', '29', '38', '47', '56', '65', '74', '92', '94', '100', '50', '64', '71', '78', '39', '95', '46', '53', '60', '67', '88', '13', '22', '31', '40', '49', '58', '41', '42', '43', '44', '30', '54', '15', '36', '72', '86', '96', '18', '19', '20', '21', '23', '24', '25', '26', '27', '28', '73', '98', '82', '52', '34', '66', '70', '68', '84', '37', '61', '11', '14'

# Generate a new sequence of any length with the pre-trained LLM


In [2]:
# generate a new sentence of any length
#      prompt_seq = [ 2, 4, 6, 8 ] 
#                        ------- e.g. batch_length=3
#     context_seq = [ 4, 6, 8 ]
#           score = [ v, v, v ] where v is the score vector over the dictionary of tokens
#                           -   the last score vector is extracted to predict the next token
# prob_next_token = softmax( v ) is the probability of the next token over the dictionary 
# next_token = sample( prob_next_token ), where sample = max probability or Bernoulli sampling
# new gen_seq = [ 2, 4, 6, 8 ] + [10] = [ 2, 4, 6, 8, 10]
# auto-regressive prediction => repeat the process 

# generate a new sentence of any length
def generate(LLMnet, prompt_seq, max_length_gen_seq):
    LLMnet.eval()
    with torch.no_grad():
        gen_seq = prompt_seq # an initial sequence (a.k.a. prompt) to generate a longer sequence with LLMnet
        for k in range(max_length_gen_seq):
            context_seq = gen_seq[:,-batch_length:] # size=[1, <=batch_length]
            score = LLMnet(context_seq) # size=[1, batch_length, num_tokens]
            score_last_token = score[:,-1,:].squeeze(dim=1) # size=[1, num_tokens]
            prob_last_token = torch.softmax(score_last_token, dim=1) # size=[1, num_tokens]
            #idx_next_token = torch.multinomial(prob_last_token, num_samples=1) # size=[1,1]
            idx_next_token = torch.max(prob_last_token, dim=1).indices[0].view(1,1) # size=[1,1]
            gen_seq = torch.cat((gen_seq, idx_next_token), dim=1) # append next token, (size=[1, num_tokens+1]
    return gen_seq

# use pre-trained network to generate a new sequence from a prompt sequence
#  prompt_seq = 25 28 31 34 37 40
#     gen_seq =             43 46 49 52 55 58 61 64 67 70 73 76 ... -- Does it generate the correct response ? 
prompt_seq_str = '25 28 31 34 37 40'
print('prompt_seq_str :', prompt_seq_str)
prompt_seq_ind = [str(i) for i in prompt_seq_str.split()] # convert a string into seq of tokens (w/ string type)
prompt_seq_ind = func_tokens2indices(prompt_seq_ind) # convert from token (str) to index (int)
prompt_seq_ind = torch.tensor(prompt_seq_ind).unsqueeze(0).to(device) # convert to pytorch
# print('prompt_seq_ind :', prompt_seq_ind)
gen_seq = generate(SSL_LLMnet, prompt_seq_ind, max_length_gen_seq=batch_length)[0][prompt_seq_ind[0].size(0):]
gen_seq_str = func_tokens2str(func_indices2tokens(gen_seq.tolist()))
print('generated sequence :', gen_seq_str)

print('-'*50)

#  prompt_seq = generate an arithmetic series with 6 terms starting with value 25 and common difference 3
#     gen_seq (wanted solution) =  25, 28, 31, 34, 37, 40 -- Does it generate the correct response ?
prompt_seq_str = 'generate an arithmetic series with 6 terms starting with value 25 and common difference 3'
print('prompt_seq_str :', prompt_seq_str)
prompt_seq_ind = [str(i) for i in prompt_seq_str.split()] # convert a string into seq of tokens (w/ string type)
prompt_seq_ind = func_tokens2indices(prompt_seq_ind) # convert from token (str) to index (int)
prompt_seq_ind = torch.tensor(prompt_seq_ind).unsqueeze(0).to(device) # convert to pytorch
# print('prompt_seq_ind :', prompt_seq_ind)
gen_seq = generate(SSL_LLMnet, prompt_seq_ind, max_length_gen_seq=batch_length)[0][prompt_seq_ind[0].size(0):]
gen_seq_str = func_tokens2str(func_indices2tokens(gen_seq.tolist()))
print('generated sequence :', gen_seq_str)


prompt_seq_str : 25 28 31 34 37 40
generated sequence : 43 46 49 52 55 58 61 <SEP> 75 76 77 78 79 80 81 82 83 84 85 86 87 88 <SEP> 90 97 <SEP> 61 67 73 79 85 91 97 <SEP> 13 19 25 31 37 43
--------------------------------------------------
prompt_seq_str : generate an arithmetic series with 6 terms starting with value 25 and common difference 3
generated sequence : <SEP> 3 12 21 30 39 48 57 66 75 84 93 <SEP> 2 7 12 17 22 27 32 37 42 47 52 57 62 67 <SEP> 20 23 26 29 32 35 38 41 44 47 50 53
